In [1]:
import pandas as pd

# 1. Download and load the dataset directly
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

# 2. Check for missing values
print("Missing values before preparation:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\n---")

# 3. Identify categorical (object) and numerical columns
categorical_cols = list(df.dtypes[df.dtypes == 'object'].index)
numerical_cols = list(df.dtypes[df.dtypes != 'object'].index)

# 4. Replace missing values
# For categorical features, replace with 'NA'
df[categorical_cols] = df[categorical_cols].fillna('NA')

# For numerical features, replace with 0.0
df[numerical_cols] = df[numerical_cols].fillna(0.0)

# 5. Verify the missing values are handled
print("\nMissing values after preparation:")
print(df.isnull().sum().max())

Missing values before preparation:
lead_source          148
industry             240
employment_status    194
location             208
annual_income        369
lead_score            35
dtype: int64

---

Missing values after preparation:
0


In [2]:
from sklearn.model_selection import train_test_split

# 1. Split the data into full_train and test (80%/20%)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

# 2. Split the full_train into train and validation (75%/25% of the 80%, meaning 60%/20% overall)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

# 3. Reset indices to keep the dataframes clean
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# 4. Verify the splits
print(f"Train size: {len(df_train)}")
print(f"Validation size: {len(df_val)}")
print(f"Test size: {len(df_test)}")

Train size: 3000
Validation size: 1000
Test size: 1000


In [3]:
# Find the most frequent observation in the 'industry' column
industry_mode = df['industry'].mode()[0]

print(f"The most frequent industry is: {industry_mode}")

# Optional: To see the counts of all categories to verify
print("\nValue counts for industry:")
print(df['industry'].value_counts().head())

The most frequent industry is: technology

Value counts for industry:
industry
technology    1173
retail         925
healthcare     840
finance        720
education      639
Name: count, dtype: int64


In [5]:
# 1. Safely select ONLY the actual numerical columns we identified earlier
numerical_features = [
    'lead_score', 
    'number_of_courses_viewed', 
    'interaction_count', 
    'annual_income'
]
correlation_matrix = df[numerical_features].corr()

# 2. Extract and print the specific pairs for Question 2
pairs_to_check = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

print("Correlation coefficients for the specified pairs:")
for feat1, feat2 in pairs_to_check:
    corr_value = correlation_matrix.loc[feat1, feat2]
    print(f"- {feat1} & {feat2}: {round(corr_value, 4)}")

# 3. Split the data using random_state=42 (as requested for Question 2)
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

# Reset indices
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# 4. Extract the target variable and remove it from the feature DataFrames
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

df_train = df_train.drop(columns=['converted'])
df_val = df_val.drop(columns=['converted'])
df_test = df_test.drop(columns=['converted'])

print("\nData splitting complete. 'converted' column removed from feature dataframes.")

Correlation coefficients for the specified pairs:
- interaction_count & lead_score: 0.9157
- number_of_courses_viewed & lead_score: 0.7572
- number_of_courses_viewed & interaction_count: 0.7216
- annual_income & interaction_count: 0.1228

Data splitting complete. 'converted' column removed from feature dataframes.


In [7]:
from sklearn.metrics import mutual_info_score

# 1. Define the categorical features
categorical_features = ['industry', 'location', 'lead_source', 'employment_status']

mi_scores = {}

# 2. Calculate the mutual information score for each feature
for feature in categorical_features:
    # Add .astype(str) to ensure uniform data types and prevent the sorting error
    score = mutual_info_score(df_train[feature].astype(str), y_train)
    mi_scores[feature] = round(score, 2)

# 3. Print the results
print("Mutual Information Scores:")
for feature, score in mi_scores.items():
    print(f"- {feature}: {score}")

# 4. Identify the feature with the highest score
best_feature = max(mi_scores, key=mi_scores.get)
print(f"\nFeature with the biggest Mutual Information score: {best_feature}")

Mutual Information Scores:
- industry: 0.0
- location: 0.0
- lead_source: 0.03
- employment_status: 0.02

Feature with the biggest Mutual Information score: lead_source


In [8]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Convert DataFrames to lists of dictionaries for DictVectorizer
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

# 2. Apply one-hot-encoding
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

# 3. Train the logistic regression model with the specified parameters
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# 4. Predict binary class labels on the validation set
y_pred_val = model.predict(X_val)

# 5. Calculate and round the accuracy score
accuracy = accuracy_score(y_val, y_pred_val)
print(f"Validation Accuracy: {round(accuracy, 2)}")

Validation Accuracy: 0.65


In [9]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Get the base accuracy with all features
features = list(df_train.columns)

dv = DictVectorizer(sparse=False)
X_train_base = dv.fit_transform(df_train[features].to_dict(orient='records'))
X_val_base = dv.transform(df_val[features].to_dict(orient='records'))

base_model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
base_model.fit(X_train_base, y_train)

base_accuracy = accuracy_score(y_val, base_model.predict(X_val_base))
print(f"Base Accuracy (All Features): {base_accuracy:.5f}\n")

# 2. Test removing specific features
features_to_check = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
differences = {}

for feature in features_to_check:
    # Create a subset of features without the current one
    subset = features.copy()
    subset.remove(feature)
    
    # Prepare the data
    dv_sub = DictVectorizer(sparse=False)
    X_train_sub = dv_sub.fit_transform(df_train[subset].to_dict(orient='records'))
    X_val_sub = dv_sub.transform(df_val[subset].to_dict(orient='records'))
    
    # Train the new model
    model_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_sub.fit(X_train_sub, y_train)
    
    # Calculate the new accuracy
    acc_sub = accuracy_score(y_val, model_sub.predict(X_val_sub))
    
    # Calculate the raw difference
    diff = base_accuracy - acc_sub
    differences[feature] = diff
    
    print(f"Without {feature:<25} | Acc: {acc_sub:.5f} | Diff: {diff:.5f}")

# 3. Find the feature with the smallest absolute difference (least impact on the model)
# "Least useful" means removing it changes the score the least (closest to 0)
least_useful_feature = min(differences, key=lambda k: abs(differences[k]))
print(f"\nFeature with the smallest difference (least useful): {least_useful_feature}")

Base Accuracy (All Features): 0.64500

Without lead_source               | Acc: 0.64200 | Diff: 0.00300
Without number_of_courses_viewed  | Acc: 0.64300 | Diff: 0.00200
Without interaction_count         | Acc: 0.60100 | Diff: 0.04400

Feature with the smallest difference (least useful): number_of_courses_viewed


In [10]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Prepare the full feature set (same as Q4)
features = list(df_train.columns)

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
X_val = dv.transform(df_val[features].to_dict(orient='records'))

# 2. Define the C values to test
C_values = [0.000001, 0.00001, 0.0001, 0.001]
results = {}

# 3. Train a model for each C and record the accuracy
for C in C_values:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    
    # Round to 3 decimal digits as instructed
    acc = round(accuracy_score(y_val, y_pred), 3)
    results[C] = acc
    
    print(f"C = {C:<8} | Accuracy: {acc:.3f}")

# 4. Find the best C (handling ties by picking the smallest C)
best_c = None
best_acc = -1

# Because we iterate through C_values in ascending order, 
# the first time we see the max accuracy, it will automatically be the smallest C.
for C in C_values:
    if results[C] > best_acc:
        best_acc = results[C]
        best_c = C

print(f"\nBest C: {best_c}")

C = 1e-06    | Accuracy: 0.598
C = 1e-05    | Accuracy: 0.598
C = 0.0001   | Accuracy: 0.613
C = 0.001    | Accuracy: 0.645

Best C: 0.001
